In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np


def short_node_label(raw):
    text = str(raw)
    if "Root" in text:
        return "Root"
    import re

    m = re.search(r"LE(\d+).*CS\1", text)
    if m:
        return f"LE/CS{m.group(1)}"
    m = re.search(r"OT(\d+)_m(\d+)", text)
    if m:
        return f"OT{m.group(1)}\nm{m.group(2)}"
    m = re.search(r"OT(\d+)-(\d+) and R", text)
    if m:
        return f"OT{m.group(1)}-{m.group(2)}\n+R"
    m = re.search(r"OT(\d+) and R", text)
    if m:
        return f"OT{m.group(1)}\n+R"
    m = re.search(r"OT(\d+)-(\d+) and F", text)
    if m:
        return f"OT{m.group(1)}-{m.group(2)}\n+F"
    m = re.search(r"OT(\d+)-(\d+)", text)
    if m:
        return f"OT{m.group(1)}-{m.group(2)}"
    if "F modes" in text:
        return "F\nmodes"
    m = re.search(r"OT(\d+).*modes", text)
    if m:
        return f"OT{m.group(1)}\nmodes"
    m = re.search(r"OT(\d+)", text)
    if m:
        return f"OT{m.group(1)}"
    if re.search(r"'R'", text):
        return "R"
    m = re.search(r"F(\d+)", text)
    if m:
        return f"F{m.group(1)}"
    return re.sub(r"[\[\]\(\)',]", "", text).replace("TreeX", "").strip() or text


def tree_layout(children, root=0):
    leaf_counts = {}

    def count_leaves(node):
        if node in leaf_counts:
            return leaf_counts[node]
        if len(children[node]) == 0:
            leaf_counts[node] = 1
        else:
            leaf_counts[node] = sum(count_leaves(child) for child in children[node])
        return leaf_counts[node]

    count_leaves(root)
    pos = {}
    max_depth = 0

    def assign(node, left, right, depth):
        nonlocal max_depth
        max_depth = max(max_depth, depth)
        pos[node] = ((left + right) / 2.0, -float(depth))
        cursor = left
        total = sum(leaf_counts[child] for child in children[node])
        for child in children[node]:
            width = (right - left) * leaf_counts[child] / total if total else 0.0
            assign(child, cursor, cursor + width, depth + 1)
            cursor += width

    assign(root, 0.0, float(leaf_counts[root]), 0)
    return pos, leaf_counts[root], max_depth


def format_entropy(value):
    if abs(value) < 1e-12:
        return "0"
    if abs(value) < 1e-3:
        return f"{value:.1e}"
    return f"{value:.3f}".rstrip("0").rstrip(".")


def format_bond_dim(value):
    try:
        return str(int(value))
    except (TypeError, ValueError):
        return str(value)


def format_edge_label(entropy, bond_dim):
    return f"{format_entropy(entropy)}\nχ={format_bond_dim(bond_dim)}"


def plot_bond_entropy(npz_path, time_fs, save_path=None, show=True):
    npz_path = Path(npz_path).expanduser()
    npz = np.load(npz_path, allow_pickle=True)
    times = np.asarray(npz["time_fs"], dtype=float)
    if not (times.min() <= time_fs <= times.max()):
        raise ValueError(f"time_fs must be between {times.min()} and {times.max()}, got {time_fs}")

    time_idx = int(np.argmin(np.abs(times - time_fs)))
    actual_time = float(times[time_idx])
    raw_labels = [str(x) for x in npz["dof_strs"]]
    adj = np.asarray(npz["adj_matrix"])
    bond_entropy = np.asarray(npz["S_bond_entropy"], dtype=float)[time_idx]
    bond_dims = np.asarray(npz["bond_dims"], dtype=object)[time_idx]

    children = {idx: list(np.flatnonzero(adj[idx])) for idx in range(len(raw_labels))}
    edges = [(parent, child) for parent, child_list in children.items() for child in child_list]
    edge_entropies = np.array([bond_entropy[child] for _, child in edges], dtype=float)
    edge_bond_dims = [bond_dims[child] for _, child in edges]

    pos, n_leaves, max_depth = tree_layout(children, root=0)
    vmax = max(float(np.nanmax(edge_entropies)), 1e-12)
    norm = mpl.colors.Normalize(vmin=0.0, vmax=vmax)
    cmap = plt.get_cmap("magma")

    fig_w = max(18, n_leaves * 0.24)
    fig_h = max(10, (max_depth + 1) * 0.85)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), constrained_layout=True)

    for (parent, child), entropy, bond_dim in zip(edges, edge_entropies, edge_bond_dims):
        x0, y0 = pos[parent]
        x1, y1 = pos[child]
        ax.plot(
            [x0, x1], [y0, y1],
            color=cmap(norm(entropy)),
            linewidth=1.15,
            solid_capstyle="round",
            zorder=1,
        )

        xm, ym = (x0 + x1) / 2.0, (y0 + y1) / 2.0
        dx, dy = x1 - x0, y1 - y0
        length = float(np.hypot(dx, dy))
        if length > 0:
            nx, ny = -dy / length, dx / length
            if ny < 0:
                nx, ny = -nx, -ny
        else:
            nx, ny = 0.0, 1.0
        ax.text(
            xm + 0.08 * nx, ym + 0.08 * ny, format_edge_label(float(entropy), bond_dim),
            ha="center", va="center",
            fontsize=4.2,
            color="#111111",
            bbox={"boxstyle": "round,pad=0.08", "facecolor": "white", "edgecolor": "none", "alpha": 0.72},
            zorder=2,
        )

    for idx, raw in enumerate(raw_labels):
        x, y = pos[idx]
        ax.text(
            x, y, short_node_label(raw),
            ha="center", va="center",
            fontsize=5.2,
            linespacing=0.92,
            bbox={"boxstyle": "round,pad=0.18", "facecolor": "white", "edgecolor": "#222222", "linewidth": 0.35},
            zorder=3,
        )

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.018, pad=0.01)
    cbar.set_label("Bond entropy S")

    ax.set_title(
        f"Singleset TreeX bond entropy at t={actual_time:g} fs\n"
        f"edge color = entropy of the child-node bond; source npz: {npz_path.name}",
        fontsize=12,
    )
    ax.set_axis_off()
    ax.margins(x=0.02, y=0.06)

    if show:
        plt.show()
    if save_path is None:
        save_path = Path.cwd() / f"{npz_path.stem}_bond_entropy_t{actual_time:g}fs.pdf"
    save_path = Path(save_path)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to {save_path}")
    return fig, ax





In [ ]:
NPZ_PATH = Path("/curie-home/zengjj/Renormalizer/multiset_ttn/P3HT:PCBM/singleset_treeX/p3ht_ttns_treeX_bond_entropy_32.npz")
TIME_FS = 200

plot_bond_entropy(NPZ_PATH, TIME_FS)